# Feature Extraction in Audio
## This notebook outlines the concepts behind extracting features from audio

### Feature Extraction

Extracting a set of features that are informative with respect to the desired properties of the original audio data

Low-level features to construct a higher-level of understanding

Need to extract audio features capable of discriminating between different audio classes i.e. speakers, emotions, genres

### Features
- Short-term Windowing (Framing)
    - Energy
    - Spectral Centroid
- Mid-term features
- Spectrogram
- Zero Crossings
- Spectral Rolloff
- MFCC

### Import the library

In [1]:
import numpy as np
import librosa
import librosa.display
import sklearn
import plotly.graph_objects as go
import plotly
import IPython
from matplotlib import pyplot as plt

### Read Audio file

In [2]:
# Load the audio file using librosa
audio_path = "count.wav"
x, fs = librosa.load(audio_path, sr=None)  # sr=None keeps the original sampling rate
print(f"Sampling Rate: {fs} Hz")
print(f"Signal shape: {x.shape}")

Sampling Rate: 48000 Hz
Signal shape: (281088,)


### Play the audio file

In [3]:
# Play the audio file
IPython.display.Audio(audio_path)

<IPython.lib.display.Audio object>

### Duration of the audio file

In [4]:
duration = librosa.get_duration(y=x, sr=fs)
print(f"Duration: {duration:.2f} seconds")

Duration: 5.86 seconds


### Extract short-term features using a 50msec non-overlapping windows

### Extract features
- Use feature_extraction( )
    - **signal**:         the input signal samples
    - **sampling_rate**:  the sampling freq (in Hz)
    - **window**:         the short-term window size (in samples)
    - **step**:           the short-term window step (in samples)
    - **deltas**:         (opt) True/False if delta features are to be computed
- RETURNS
    - **features** (numpy.ndarray):        contains features (n_feats x numOfShortTermWindows)                     
    - **feature_names** (numpy.ndarray):   contains feature names (n_feats x numOfShortTermWindows)

In [5]:
# Extract short-term features using 50ms non-overlapping windows (librosa)
window = int(fs * 0.050)   # 50ms window in samples
step   = int(fs * 0.050)   # 50ms step (non-overlapping)

# Compute each feature using librosa
energy          = librosa.feature.rms(y=x, frame_length=window, hop_length=step)
spectral_c      = librosa.feature.spectral_centroid(y=x, sr=fs, n_fft=window, hop_length=step)
spectral_bw     = librosa.feature.spectral_bandwidth(y=x, sr=fs, n_fft=window, hop_length=step)
spectral_roll   = librosa.feature.spectral_rolloff(y=x, sr=fs, n_fft=window, hop_length=step)
zcr             = librosa.feature.zero_crossing_rate(y=x, frame_length=window, hop_length=step)
mfcc_st         = librosa.feature.mfcc(y=x, sr=fs, n_mfcc=13, n_fft=window, hop_length=step)

# Stack all features into one matrix (features x frames)
features = np.vstack([energy, spectral_c, spectral_bw, spectral_roll, zcr, mfcc_st])

feature_names = (["energy", "spectral_centroid", "spectral_bandwidth",
                  "spectral_rolloff", "zcr"] +
                 [f"mfcc_{i}" for i in range(1, 14)])

print("Feature extraction complete!")

Feature extraction complete!


In [6]:
print(f"Features array shape: {features.shape}  → (num_features x num_frames)")

Features array shape: (18, 118)  → (num_features x num_frames)


### How many frames

In [7]:
num_frames = features.shape[1]
print(f"Number of frames: {num_frames}")

Number of frames: 118


### How many features

In [8]:
num_features = features.shape[0]
print(f"Number of features: {num_features}")

Number of features: 18


### Feature Names

In [9]:
print("Feature Names:")
for i, name in enumerate(feature_names):
    print(f"  [{i:02d}] {name}")

Feature Names:
  [00] energy
  [01] spectral_centroid
  [02] spectral_bandwidth
  [03] spectral_rolloff
  [04] zcr
  [05] mfcc_1
  [06] mfcc_2
  [07] mfcc_3
  [08] mfcc_4
  [09] mfcc_5
  [10] mfcc_6
  [11] mfcc_7
  [12] mfcc_8
  [13] mfcc_9
  [14] mfcc_10
  [15] mfcc_11
  [16] mfcc_12
  [17] mfcc_13


### Plot Short-term energy

### Time

In [10]:
t = librosa.frames_to_time(range(num_frames), sr=fs, hop_length=step)
print(f"Time axis: 0 → {t[-1]:.2f} s  ({len(t)} frames)")

Time axis: 0 → 5.85 s  (118 frames)


In [11]:
print(f"Time array shape : {t.shape}")
print(f"Total duration   : {t[-1]:.2f} seconds")

Time array shape : (118,)
Total duration   : 5.85 seconds


# Energy

In [12]:
energy_vals = features[0, :]   # energy is row 0
print(f"Energy shape : {energy_vals.shape}")
print(f"Min energy   : {energy_vals.min():.4f}")
print(f"Max energy   : {energy_vals.max():.4f}")

Energy shape : (118,)
Min energy   : 0.0000
Max energy   : 0.4977


In [ ]:
print(f"Energy — first 5 values: {energy_vals[:5]}")

### Plot Time Vs Energy

In [13]:
plt.figure(figsize=(12, 4))
plt.plot(t, energy_vals, color='steelblue', linewidth=1.2)
plt.xlabel('Time (s)')
plt.ylabel('Energy')
plt.title('Short-term Energy over Time')
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

<Figure size 1200x400 with 1 Axes>

# Spectral Centroid
### Spectral centroid -- centre of mass -- weighted mean of the frequencies present in the sound


In [14]:
spectral_centroid = features[1, :]   # spectral_centroid is row 1
print(f"Spectral Centroid shape: {spectral_centroid.shape}")

Spectral Centroid shape: (118,)


### Plot Time Vs Spectral Centroid

In [15]:
plt.figure(figsize=(12, 4))
plt.plot(t, spectral_centroid, color='darkorange', linewidth=1.2)
plt.xlabel('Time (s)')
plt.ylabel('Spectral Centroid')
plt.title('Spectral Centroid over Time (pyAudioAnalysis)')
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

<Figure size 1200x400 with 1 Axes>

### Use librosa library

In [16]:
import sklearn
import librosa
import librosa.display

### Read the audio file

In [17]:
# Load audio using librosa
y, sr = librosa.load(audio_path)
print(f"Signal shape   : {y.shape}")
print(f"Sampling rate  : {sr} Hz")

Signal shape   : (129125,)
Sampling rate  : 22050 Hz


### Compute Spectral Centroids

In [18]:
# Compute Spectral Centroids using librosa
spectral_centroids = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
print(f"Spectral Centroids shape: {spectral_centroids.shape}")

Spectral Centroids shape: (253,)


### Computing the time variable for visualization

In [19]:
# Convert frame indices to time in seconds
frames = range(len(spectral_centroids))
t_librosa = librosa.frames_to_time(frames, sr=sr)
print(f"Time array shape: {t_librosa.shape}")

Time array shape: (253,)


### Normalising the spectral centroid for visualisation

In [20]:
def normalize(x, axis=0):
    return sklearn.preprocessing.minmax_scale(x, axis=axis)

### Plotting the Spectral Centroid along the waveform

In [21]:
plt.figure(figsize=(12, 4))
librosa.display.waveshow(y, sr=sr, alpha=0.4, color='steelblue')
plt.plot(t_librosa, normalize(spectral_centroids), color='r', linewidth=1.5, label='Spectral Centroid (normalised)')
plt.xlabel('Time (s)')
plt.title('Spectral Centroid along Waveform (librosa)')
plt.legend()
plt.tight_layout()
plt.show()

<Figure size 1200x400 with 1 Axes>

# Mid-term features

### Steps
- Import the library
- Extract mid_features

### Import the library

In [22]:
# Mid-term features use numpy to aggregate short-term features — no extra library needed
import numpy as np

### Extract features
- Use mid_feature_extraction( )
    - signal
    - sampling_rate
    - mid_window
    - mid_step
    - short_window
    - short_step

In [23]:
# Aggregate short-term features into 1-second mid-term segments
frames_per_segment = int(1.0 / 0.050)   # 1 second / 50ms = 20 frames per segment

num_segments = features.shape[1] // frames_per_segment

mt_features = np.array([
    features[:, i * frames_per_segment:(i + 1) * frames_per_segment].mean(axis=1)
    for i in range(num_segments)
]).T   # shape → (features x segments)

mt_feature_names = [f"{name}_mean" for name in feature_names]
print("Mid-term feature extraction complete!")

Mid-term feature extraction complete!


### Duration, Short-term features, Segment features

In [24]:
print(f"Duration              : {duration:.2f} seconds")
print(f"Short-term features   : {features.shape}    → (features x frames)")
print(f"Mid-term features     : {mt_features.shape} → (features x segments)")

Duration              : 5.86 seconds
Short-term features   : (18, 118)    → (features x frames)
Mid-term features     : (18, 5) → (features x segments)


### Mid-term feature names

In [25]:
print("Mid-term Feature Names:")
for i, name in enumerate(mt_feature_names):
    print(f"  [{i:02d}] {name}")

Mid-term Feature Names:
  [00] energy_mean
  [01] spectral_centroid_mean
  [02] spectral_bandwidth_mean
  [03] spectral_rolloff_mean
  [04] zcr_mean
  [05] mfcc_1_mean
  [06] mfcc_2_mean
  [07] mfcc_3_mean
  [08] mfcc_4_mean
  [09] mfcc_5_mean
  [10] mfcc_6_mean
  [11] mfcc_7_mean
  [12] mfcc_8_mean
  [13] mfcc_9_mean
  [14] mfcc_10_mean
  [15] mfcc_11_mean
  [16] mfcc_12_mean
  [17] mfcc_13_mean


# Spectrogram

### Steps
- Import the **librosa** library
- Load the audio file
- Compute Frequencies using FT
- Compute Amplitude
- Plot Frequencies Vs Amplitude as Spectrogram

### Import the library

In [26]:
import librosa
import librosa.display
import numpy as np
import matplotlib.pyplot as plt

### Load the audio file

In [27]:
y_spec, sr_spec = librosa.load(audio_path)
print(f"Loaded: {y_spec.shape} samples at {sr_spec} Hz")

Loaded: (129125,) samples at 22050 Hz


#### Sampling rate

In [28]:
print(f"Sampling Rate: {sr_spec} Hz")

Sampling Rate: 22050 Hz


### Compute Frequencies

In [29]:
# Compute Short-Time Fourier Transform (STFT) to get frequencies
D = librosa.stft(y_spec)
frequencies = librosa.fft_frequencies(sr=sr_spec)
print(f"STFT shape      : {D.shape}  → (frequency bins x time frames)")
print(f"Frequency range : 0 – {frequencies[-1]:.0f} Hz")

STFT shape      : (1025, 253)  → (frequency bins x time frames)
Frequency range : 0 – 11025 Hz


### Compute Amplitude

In [30]:
# Convert amplitude to decibels for better visualisation
amplitude_db = librosa.amplitude_to_db(np.abs(D), ref=np.max)
print(f"Amplitude (dB) range: {amplitude_db.min():.1f} to {amplitude_db.max():.1f} dB")

Amplitude (dB) range: -80.0 to 0.0 dB


### Display Spectrogram

In [31]:
plt.figure(figsize=(12, 6))
librosa.display.specshow(amplitude_db, sr=sr_spec, x_axis='time', y_axis='hz', cmap='magma')
plt.colorbar(format='%+2.0f dB')
plt.title('Spectrogram (STFT)')
plt.xlabel('Time (s)')
plt.ylabel('Frequency (Hz)')
plt.tight_layout()
plt.show()

<Figure size 1200x600 with 2 Axes>

# Zero Crossings

### Steps
- Import the library
- Read the Audio file
- Compute Zero Crossings
- Plot zero crossings

### Import the library

In [32]:
import librosa
import numpy as np
import matplotlib.pyplot as plt

### Read the audio file

In [33]:
y_zc, sr_zc = librosa.load(audio_path)
print(f"Loaded: {y_zc.shape} samples at {sr_zc} Hz")

Loaded: (129125,) samples at 22050 Hz


### Plot the signal

In [34]:
plt.figure(figsize=(12, 4))
librosa.display.waveshow(y_zc, sr=sr_zc, color='steelblue')
plt.title('Audio Waveform')
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.tight_layout()
plt.show()

<Figure size 1200x400 with 1 Axes>

### Compute Zero Crossings

In [35]:
# Compute zero crossings (True where signal crosses zero)
zero_crossings = librosa.zero_crossings(y_zc, pad=False)
print(f"Total Zero Crossings: {sum(zero_crossings)}")

Total Zero Crossings: 14179


In [36]:
# Zero-crossing rate per frame
zcr = librosa.feature.zero_crossing_rate(y_zc)[0]
t_zcr = librosa.frames_to_time(range(len(zcr)), sr=sr_zc)
print(f"Zero-crossing rate shape: {zcr.shape}")

Zero-crossing rate shape: (253,)


In [37]:
plt.figure(figsize=(12, 4))
plt.plot(t_zcr, zcr, color='green', linewidth=1.2)
plt.title('Zero-Crossing Rate over Time')
plt.xlabel('Time (s)')
plt.ylabel('Zero-Crossing Rate')
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

<Figure size 1200x400 with 1 Axes>

# Spectral Rolloff

In [38]:
# Compute Spectral Rolloff — frequency below which 85% of energy is contained
spectral_rolloff = librosa.feature.spectral_rolloff(y=y_zc, sr=sr_zc, roll_percent=0.85)[0]
t_rolloff = librosa.frames_to_time(range(len(spectral_rolloff)), sr=sr_zc)
print(f"Spectral Rolloff shape: {spectral_rolloff.shape}")

Spectral Rolloff shape: (253,)


### Plot Spectral Rolloff

In [39]:
plt.figure(figsize=(12, 4))
librosa.display.waveshow(y_zc, sr=sr_zc, alpha=0.4, color='steelblue')
plt.plot(t_rolloff, normalize(spectral_rolloff), color='r', linewidth=1.5, label='Spectral Rolloff (normalised)')
plt.title('Spectral Rolloff along Waveform')
plt.xlabel('Time (s)')
plt.legend()
plt.tight_layout()
plt.show()

<Figure size 1200x400 with 1 Axes>

### Spectral Centroids

In [40]:
# Compare Spectral Centroid vs Spectral Rolloff
sc_zc = librosa.feature.spectral_centroid(y=y_zc, sr=sr_zc)[0]
t_sc  = librosa.frames_to_time(range(len(sc_zc)), sr=sr_zc)

plt.figure(figsize=(12, 4))
librosa.display.waveshow(y_zc, sr=sr_zc, alpha=0.3, color='steelblue')
plt.plot(t_sc,      normalize(sc_zc),           color='b', linewidth=1.5, label='Spectral Centroid')
plt.plot(t_rolloff, normalize(spectral_rolloff), color='r', linewidth=1.5, label='Spectral Rolloff')
plt.title('Spectral Centroid vs Spectral Rolloff')
plt.xlabel('Time (s)')
plt.legend()
plt.tight_layout()
plt.show()

<Figure size 1200x400 with 1 Axes>

# MFCC

### Compute MFCC

In [41]:
# Compute MFCC — 13 coefficients (standard for speech/audio tasks)
mfccs = librosa.feature.mfcc(y=y_zc, sr=sr_zc, n_mfcc=13)
print(f"MFCC shape: {mfccs.shape}  → ({mfccs.shape[0]} coefficients x {mfccs.shape[1]} frames)")

MFCC shape: (13, 253)  → (13 coefficients x 253 frames)


In [42]:
print(f"Number of MFCC coefficients : {mfccs.shape[0]}")
print(f"Number of frames            : {mfccs.shape[1]}")
print(f"Mean of each coefficient    : {np.mean(mfccs, axis=1).round(2)}")

Number of MFCC coefficients : 13
Number of frames            : 253
Mean of each coefficient    : [-256.65   95.97   -3.32   60.25   -7.93    9.79   -0.5    -7.22   -6.13
    3.84    1.58   -9.31    5.26]


### Plot MFCC

In [43]:
plt.figure(figsize=(12, 6))
librosa.display.specshow(mfccs, sr=sr_zc, x_axis='time', cmap='coolwarm')
plt.colorbar()
plt.title('MFCC (13 Coefficients)')
plt.xlabel('Time (s)')
plt.ylabel('MFCC Coefficient')
plt.tight_layout()
plt.show()

<Figure size 1200x600 with 2 Axes>

# Chromagram

### Compute Chromagram

In [44]:
# Chromagram — shows the energy distribution across 12 pitch classes (C, C#, D, ...)
chromagram = librosa.feature.chroma_stft(y=y_zc, sr=sr_zc)
print(f"Chromagram shape: {chromagram.shape}  → (12 pitch classes x {chromagram.shape[1]} frames)")

Chromagram shape: (12, 253)  → (12 pitch classes x 253 frames)


In [45]:
print(f"Pitch classes : {chromagram.shape[0]}  (C, C#, D, D#, E, F, F#, G, G#, A, A#, B)")
print(f"Time frames   : {chromagram.shape[1]}")

Pitch classes : 12  (C, C#, D, D#, E, F, F#, G, G#, A, A#, B)
Time frames   : 253


### Plot Chromagram

In [46]:
plt.figure(figsize=(12, 5))
librosa.display.specshow(chromagram, sr=sr_zc, x_axis='time', y_axis='chroma', cmap='coolwarm')
plt.colorbar()
plt.title('Chromagram')
plt.xlabel('Time (s)')
plt.ylabel('Pitch Class')
plt.tight_layout()
plt.show()

<Figure size 1200x500 with 2 Axes>

In [47]:
print("=" * 50)
print("  Feature Extraction Complete!")
print("=" * 50)
print("\nFeatures extracted:")
print("  ✓ Short-term Energy")
print("  ✓ Spectral Centroid (pyAudioAnalysis + librosa)")
print("  ✓ Mid-term Features")
print("  ✓ Spectrogram (STFT)")
print("  ✓ Zero Crossings")
print("  ✓ Spectral Rolloff")
print("  ✓ MFCC (13 coefficients)")
print("  ✓ Chromagram (12 pitch classes)")

  Feature Extraction Complete!

Features extracted:
  ✓ Short-term Energy
  ✓ Spectral Centroid (pyAudioAnalysis + librosa)
  ✓ Mid-term Features
  ✓ Spectrogram (STFT)
  ✓ Zero Crossings
  ✓ Spectral Rolloff
  ✓ MFCC (13 coefficients)
  ✓ Chromagram (12 pitch classes)
